****Question1: Embedding a query****

Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

-0.31
-0.02
0.12
0.44

In [2]:
import numpy as np
from embedder import Embedder
embed = Embedder()

query = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query)

v_query.shape

2026-06-29 02:53:24.103770161 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


(384,)

**Q1: Answer**

In [3]:
v_query[0]

np.float64(-0.02058203437252893)

**Q2:Cosine similarity**

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

0.07
0.37
0.68
0.92

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
# documents[:5]

In [6]:
filename="02-vector-search/lessons/07-sqlitesearch-vector.md"
q2_content=None
for i in documents:
   if i["filename"] == filename:
      print("Found it")
      print(f"{i['filename']}")
      q2_content = i['content']

Found it
02-vector-search/lessons/07-sqlitesearch-vector.md


In [7]:
v2 = embed.encode(q2_content)
v2.shape

(384,)

**Q2: Answer**

In [8]:
score = v2.dot(v_query)
score

np.float64(0.36107027225589694)

**Q3. Chunking and search by hand**

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
len(chunks)

295

In [11]:
texts = [doc['content'] for doc in chunks]
texts[:5]

['# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language

In [12]:
X = embed.encode_batch(texts)
X.shape

(295, 384)

In [13]:
scores = X.dot(v_query)
idx = np.argmax(scores)
idx

np.int64(94)

**Q3: answer**

In [14]:
q3_filename = chunks[idx]['filename']
q3_filename

'02-vector-search/lessons/07-sqlitesearch-vector.md'

**Q4. Vector search with minsearch**

In [15]:
from minsearch import VectorSearch


vindex = VectorSearch()
vindex.fit(X, chunks)

**q4: answer**

In [16]:
query = "What metric do we use to evaluate a search engine?"
v4_query = embed.encode(query)
results = vindex.search(v4_query,num_results=1)
results

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

**Q5: Text search vs vector search**

In [17]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(chunks)

In [18]:
question = "How do I store vectors in PostgreSQL?"

text_results = index.search(
    question,
    num_results = 5
)

text= [doc['filename'] for doc in text_results]
text

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [19]:
v5_query = embed.encode("How do I store vectors in PostgreSQL?")
vector_results = vindex.search(v5_query,num_results=5)
vector = [doc['filename'] for doc in vector_results]
vector

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

**Q5: answer**

In [20]:
for i,value in enumerate(vector):
    if value not in text:
        print(value)

    

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


**Q6. Hybrid search**

In [21]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [22]:
query = "How do I give the model access to tools?"


text_results = index.search(
    query,
    num_results = 5
)
text_results

[{'start': 0,
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can

In [23]:

v6_query = embed.encode(query)
vector_results = vindex.search(v6_query,num_results=5)
vector_results

[{'start': 2000,
  'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline ag

In [24]:
results = rrf([vector_results, text_results])


**Q6: Answer**

In [25]:
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 